## Podcast Video-First
## Discovery Feed A/B Test

### Internmo Data Analytics Project

Objective- Training case study modeled on a Spotify-style product experiment. Data cleaning, analyzing and visualing the given dataset.

## 1. Data Investigating

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import os

In [2]:
# Importing Dataset Files
df1 = pd.read_csv ("C:\\Spotify_A-B_Test_project\\01_Data\\01_raw_data\\daily_user_activity.csv")
df2 = pd.read_csv ("C:\\Spotify_A-B_Test_project\\01_Data\\01_raw_data\\users_experiment_map.csv")

print("Both Excel files loaded successfully!")

Both Excel files loaded successfully!


In [3]:
print("========== Daily User Activity ==========")

print("Shape:", df1.shape)
print("Columns:", df1.columns.tolist())

display(df1.head())

print("\n========== User Experiment Map ==========")

print("Shape:", df2.shape)
print("Columns:", df2.columns.tolist())

display(df2.head())

========== Daily User Activity ==========
Shape: (19741, 6)
Columns: ['Date', 'User_ID', 'Podcast_Minutes_Listened', 'Music_Minutes_Listened', 'Discovery_Feed_Clicks', 'App_Crashes']


,Date,User_ID,Podcast_Minutes_Listened,Music_Minutes_Listened,Discovery_Feed_Clicks,App_Crashes
0,2025-03-16,U764217,24.7,83.9,0,0
1,2025-03-07,U841238,6.4,61.0,1,0
2,2025-03-12,U126809,13.6,0.0,3,0
3,2025-03-05,U863258,3.9,70.3,2,0
4,2025-03-10,U169727,24.1,0.0,0,0



========== User Experiment Map ==========
Shape: (2000, 5)
Columns: ['User_ID', 'Variant', 'Region', 'Premium_Status', 'Account_Age_Days']


,User_ID,Variant,Region,Premium_Status,Account_Age_Days
0,U428241,Treatment,LATAM,Free,236.0
1,U454990,Control,Europe,Free,770.0
2,U195228,Treatment,APAC,Premium,983.0
3,U651302,Treatment,North America,Free,314.0
4,U926541,Control,Europe,Free,430.0


In [4]:
# Checking missing values
print("Missing values - Daily User Activity:")
print(df1.isnull().sum())

print("\nMissing values - User Experiment Map:")
print(df2.isnull().sum())

# Checking Duplicates in the dataset
print("\nDuplicate rows - Daily User Activity:", df1.duplicated().sum())

print("Duplicate rows - User Experiment Map:", df2.duplicated().sum())

Missing values - Daily User Activity:
Date                          0
User_ID                       0
Podcast_Minutes_Listened    158
Music_Minutes_Listened        0
Discovery_Feed_Clicks         0
App_Crashes                   0
dtype: int64

Missing values - User Experiment Map:
User_ID              0
Variant              0
Region               0
Premium_Status       0
Account_Age_Days    30
dtype: int64

Duplicate rows - Daily User Activity: 78
Duplicate rows - User Experiment Map: 0


### 1.1 Investigate fully duplicated activity rows

In [5]:
# Fully duplicated activity rows

full_duplicates = df1[
    df1.duplicated(keep=False)
]

print("Total rows involved in full duplicates:",
      len(full_duplicates))

print("Duplicate occurrences:",
      df1.duplicated().sum())

display(full_duplicates.head(10))

Total rows involved in full duplicates: 156
Duplicate occurrences: 78


,Date,User_ID,Podcast_Minutes_Listened,Music_Minutes_Listened,Discovery_Feed_Clicks,App_Crashes
157,2025-03-05,U870167,29.1,22.4,0,0
421,2025-03-05,U578201,0.0,0.0,1,0
469,2025-03-13,U290544,7.9,14.5,5,0
652,2025-03-06,U774991,38.4,17.4,1,0
672,2025-03-16,U281174,3.5,27.6,1,0
806,2025-03-16,U811605,13.6,15.0,0,0
1124,2025-03-08,U836071,17.1,20.2,1,0
1393,2025-03-15,U643893,6.5,22.4,3,0
1468,2025-03-08,U961730,9.8,82.1,1,0
1508,2025-03-09,U139021,5.1,27.6,0,0


### 1.2 Investigate NULL Podcast_Minutes_Listened

In [6]:
# Missing podcast listening minutes

null_podcast = df1[
    df1['Podcast_Minutes_Listened'].isnull()
]

print("Total NULL Podcast Minutes:",
      len(null_podcast))

print("Percentage:",
      round(len(null_podcast) / len(df1) * 100, 2),
      "%")

display(null_podcast.head(10))

Total NULL Podcast Minutes: 158
Percentage: 0.8 %


,Date,User_ID,Podcast_Minutes_Listened,Music_Minutes_Listened,Discovery_Feed_Clicks,App_Crashes
213,2025-03-06,U474209,NaN,108.0,0,0
308,2025-03-12,U797644,NaN,40.2,1,0
353,2025-03-12,U693714,NaN,95.8,1,0
508,2025-03-08,U684609,NaN,55.7,2,0
519,2025-03-06,U170187,NaN,28.8,0,0
542,2025-03-06,U395969,NaN,0.0,3,0
639,2025-03-15,U149901,NaN,46.0,0,0
670,2025-03-06,U323842,NaN,24.9,0,0
960,2025-03-08,U1000683,NaN,39.6,1,0
1050,2025-03-11,U587503,NaN,28.0,0,0


In [7]:
# Activity patterns for records with missing podcast minutes

print(
    null_podcast[
        ['Music_Minutes_Listened',
         'Discovery_Feed_Clicks',
         'App_Crashes']
    ].describe()
)

       Music_Minutes_Listened  Discovery_Feed_Clicks  App_Crashes
count              158.000000             158.000000   158.000000
mean                58.777848               1.069620     0.037975
std                 53.707529               1.277523     0.191743
min                  0.000000               0.000000     0.000000
25%                 24.100000               0.000000     0.000000
50%                 47.750000               1.000000     0.000000
75%                 75.575000               2.000000     0.000000
max                368.000000               7.000000     1.000000


### 1.3 Investigate impossible values (more than 1,440 minutes)

In [8]:
# Identify impossible daily listening values

impossible_values = df1[
    (df1['Podcast_Minutes_Listened'] > 1440) |
    (df1['Music_Minutes_Listened'] > 1440)
]

print("Impossible activity rows:",
      len(impossible_values))

display(impossible_values)

Impossible activity rows: 13


,Date,User_ID,Podcast_Minutes_Listened,Music_Minutes_Listened,Discovery_Feed_Clicks,App_Crashes
2910,2025-03-15,U424866,2985.0,77.2,0,0
3894,2025-03-14,U192779,2753.0,77.4,0,0
5752,2025-03-12,U521399,2261.0,35.7,0,0
6360,2025-03-16,U341353,2155.0,52.3,0,0
6456,2025-03-10,U878788,2638.0,0.0,1,0
7279,2025-03-15,U424866,2985.0,77.2,0,0
8048,2025-03-04,U104042,1862.0,57.6,0,0
9640,2025-03-03,U932429,2632.0,92.0,1,0
10112,2025-03-06,U371108,2635.0,86.0,0,0
10209,2025-03-12,U423217,2369.0,16.9,2,0


In [9]:
# Identify negative listening values

negative_values = df1[
    (df1['Podcast_Minutes_Listened'] < 0) |
    (df1['Music_Minutes_Listened'] < 0)
]

print("Negative activity rows:",
      len(negative_values))

display(negative_values)

Negative activity rows: 0


,Date,User_ID,Podcast_Minutes_Listened,Music_Minutes_Listened,Discovery_Feed_Clicks,App_Crashes


### 1.4 Investigate activity rows with unknown User_ID

In [10]:
# Identify activity users missing from user map

unmatched_users = df1[
    ~df1['User_ID'].isin(df2['User_ID'])
]

print("Unmatched activity rows:",
      len(unmatched_users))

print("Unique unmatched users:",
      unmatched_users['User_ID'].nunique())

display(unmatched_users.head(10))

Unmatched activity rows: 20
Unique unmatched users: 20


,Date,User_ID,Podcast_Minutes_Listened,Music_Minutes_Listened,Discovery_Feed_Clicks,App_Crashes
684,2025-03-03,U1000880,2.4,121.0,1,0
960,2025-03-08,U1000683,NaN,39.6,1,0
1785,2025-03-16,U1000106,0.0,121.7,1,0
3445,2025-03-07,U1000951,32.0,154.1,2,0
3599,2025-03-14,U1000976,4.4,25.8,2,0
4822,2025-03-08,U1000631,13.3,0.0,0,0
5472,2025-03-11,U1000845,0.0,0.0,0,0
7649,2025-03-15,U1000364,0.0,89.2,3,0
8315,2025-03-08,U1000513,2.5,28.0,0,0
9213,2025-03-03,U1000195,27.2,199.2,4,0


### 1.5 Investigate inconsistent Region casing and whitespace

In [11]:
# Investigate original Region values

print("Unique Region values:")

print(
    df2['Region']
    .value_counts(dropna=False)
)

Unique Region values:
Region
North America      613
Europe             579
APAC               342
LATAM              300
MEA                119
 north america      13
EUROPE              10
 europe              9
NORTH AMERICA        7
 latam               3
 mea                 3
 apac                2
Name: count, dtype: int64


### 1.6 Investigate missing Account_Age_Days

In [12]:
    # Missing account age

missing_account_age = df2[
    df2['Account_Age_Days'].isnull()
]

print("Missing Account Age:",
      len(missing_account_age))

print("Percentage:",
      round(len(missing_account_age) / len(df2) * 100, 2),
      "%")

display(missing_account_age.head(10))

Missing Account Age: 30
Percentage: 1.5 %


,User_ID,Variant,Region,Premium_Status,Account_Age_Days
33,U595214,Control,Europe,Free,NaN
44,U493834,Treatment,Europe,Free,NaN
61,U104232,Control,LATAM,Free,NaN
79,U579154,Control,Europe,Premium,NaN
139,U487545,Treatment,APAC,Free,NaN
161,U351728,Control,North America,Premium,NaN
225,U258605,Control,APAC,Free,NaN
236,U493130,Control,Europe,Free,NaN
242,U181250,Control,Europe,Free,NaN
340,U613875,Control,North America,Premium,NaN


### 1.7 Investigate users with zero activity rows

In [13]:
# Find users without any activity records

active_user_ids = df2['User_ID'].unique()

zero_activity_users = df2[
    ~df2['User_ID'].isin(active_user_ids)
]

print("Total enrolled users:",
      df2['User_ID'].nunique())

print("Users with activity:",
      len(active_user_ids))

print("Users with zero activity rows:",
      len(zero_activity_users))

display(zero_activity_users.head(10))

Total enrolled users: 2000
Users with activity: 2000
Users with zero activity rows: 0


,User_ID,Variant,Region,Premium_Status,Account_Age_Days


### 1.8 Investigate rows with inconsistent region formatting

In [14]:
# Identify rows with inconsistent Region formatting

inconsistent_regions = df2[
    df2['Region'] !=
    df2['Region'].astype('string').str.strip().str.lower()
]

print("Affected rows:", len(inconsistent_regions))

display(inconsistent_regions)

Affected rows: 2000


,User_ID,Variant,Region,Premium_Status,Account_Age_Days
0,U428241,Treatment,LATAM,Free,236.0
1,U454990,Control,Europe,Free,770.0
2,U195228,Treatment,APAC,Premium,983.0
3,U651302,Treatment,North America,Free,314.0
4,U926541,Control,Europe,Free,430.0
...,...,...,...,...,...
1995,U444461,Control,Europe,Free,176.0
1996,U511748,Control,Europe,Free,711.0
1997,U487220,Control,APAC,Free,527.0
1998,U867113,Control,APAC,Free,3425.0


In [15]:
investigation_summary = pd.DataFrame({
    'Issue': [
        'Fully duplicated activity rows',
        'NULL Podcast_Minutes_Listened',
        'Impossible listening values (>1440)',
        'Negative listening values',
        'Unmatched User_ID',
        'Rows with inconsistent Region formatting',
        'Missing Account_Age_Days',
        'Users with zero activity rows'
    ],

    'Count': [
        df1.duplicated().sum(),
        df1['Podcast_Minutes_Listened'].isnull().sum(),
        len(impossible_values),
        len(negative_values),
        df1[~df1['User_ID'].isin(df2['User_ID'])].shape[0],
        len(inconsistent_regions),
        df2['Account_Age_Days'].isnull().sum(),
        len(zero_activity_users)
    ]
})

display(investigation_summary)

,Issue,Count
0,Fully duplicated activity rows,78
1,NULL Podcast_Minutes_Listened,158
2,Impossible listening values (>1440),13
3,Negative listening values,0
4,Unmatched User_ID,20
5,Rows with inconsistent Region formatting,2000
6,Missing Account_Age_Days,30
7,Users with zero activity rows,0


## Key Findings
1. Duplicate activity records (78)
The activity dataset contains 78 duplicate occurrences. These should be investigated because duplicates could inflate listening activity and affect the experiment's primary metric.

2. Missing podcast listening minutes (158)
There are 158 records where Podcast_Minutes_Listened is missing. We need to investigate whether these represent missing measurements or another type of data issue.

3. Impossible listening values (13)
Thirteen activity records contain listening values exceeding 1,440 minutes per day. These values are not logically possible and require further investigation.

4. Unmatched User IDs (20)
Twenty activity records have User_ID values that do not appear in the user experiment map. These records cannot currently be assigned to a randomized experiment group using the user map.

5. Region formatting inconsistencies (2,000)
All 2,000 user records differ from the standardized lowercase-and-trimmed format. This indicates that Region values need formatting normalization. It does not necessarily mean there are 2,000 distinct Region mismatches.

6. Missing account age (30)
Thirty users have missing Account_Age_Days values. This may affect user-level descriptive analysis and segmentation.

7. Users with zero activity rows (0)
The investigation found no enrolled users without activity records. However, we should still preserve the ITT requirement: every randomized user must remain in the analysis, including users with zero podcast listening minutes.

## 2. Data Cleaning

In [21]:
# Create backup copies of original datasets

df1_original = df1.copy()
df2_original = df2.copy()

print("Original datasets backed up successfully!")

Original datasets backed up successfully!


### 2.1 Removing fully duplicated activity rows

In [28]:
# Check the current working dataset
print("Before cleaning:", len(df1))

# Remove exact duplicates
df1 = df1.drop_duplicates().copy()

# Verify the cleaned dataset
print("After cleaning:", len(df1))
print("Remaining duplicates:", df1.duplicated().sum())

Before cleaning: 19741
After cleaning: 19663
Remaining duplicates: 0


### 2.2 Handling impossible listening values

In [30]:
# Identify impossible listening records

impossible_values = df1[
    (df1['Podcast_Minutes_Listened'] > 1440) |
    (df1['Music_Minutes_Listened'] > 1440)
]

display(impossible_values)

,Date,User_ID,Podcast_Minutes_Listened,Music_Minutes_Listened,Discovery_Feed_Clicks,App_Crashes
2910,2025-03-15,U424866,2985.0,77.2,0,0
3894,2025-03-14,U192779,2753.0,77.4,0,0
5752,2025-03-12,U521399,2261.0,35.7,0,0
6360,2025-03-16,U341353,2155.0,52.3,0,0
6456,2025-03-10,U878788,2638.0,0.0,1,0
8048,2025-03-04,U104042,1862.0,57.6,0,0
9640,2025-03-03,U932429,2632.0,92.0,1,0
10112,2025-03-06,U371108,2635.0,86.0,0,0
10209,2025-03-12,U423217,2369.0,16.9,2,0
15355,2025-03-11,U775547,2701.0,0.0,5,0


In [31]:
# Find podcast listening values greater than 1,440 minutes
impossible_mask = df1['Podcast_Minutes_Listened'] > 1440

print("Impossible values found:", impossible_mask.sum())

Impossible values found: 12


In [32]:
# Replace impossible podcast listening values with NaN
df1.loc[impossible_mask, 'Podcast_Minutes_Listened'] = np.nan

print(
    "Remaining impossible values:",
    (df1['Podcast_Minutes_Listened'] > 1440).sum()
)

print(
    "Missing podcast values after cleaning:",
    df1['Podcast_Minutes_Listened'].isna().sum()
)

Remaining impossible values: 0
Missing podcast values after cleaning: 170


### 2.3 Handling Missing values in Podcast_Minutes_Listened column

In [33]:
# Retrieve the original podcast listening values
original_podcast = df1_original[
    'Podcast_Minutes_Listened'
].reindex(df1.index)

# Identify the reason for each missing value
df1['Missing_Reason'] = np.where(
    original_podcast.isna(),
    'Originally Missing',
    np.where(
        original_podcast > 1440,
        'Impossible Value',
        'Not Missing'
    )
)

# Count missing values by reason
print(df1['Missing_Reason'].value_counts())

Missing_Reason
Not Missing           19493
Originally Missing      158
Impossible Value         12
Name: count, dtype: int64


In [34]:
# Join experiment group information
missing_analysis = df1.merge(
    df2[['User_ID', 'Variant']],
    on='User_ID',
    how='left'
)

# Calculate missing values by experiment group
missing_by_variant = missing_analysis.groupby('Variant').agg(
    Total_Records=('Podcast_Minutes_Listened', 'size'),
    Missing_Records=('Podcast_Minutes_Listened', 'count')
)

# Convert count of non-missing records into missing count
missing_by_variant['Missing_Records'] = (
    missing_analysis.groupby('Variant')['Podcast_Minutes_Listened']
    .apply(lambda x: x.isna().sum())
)

missing_by_variant['Missing_Percentage'] = (
    missing_by_variant['Missing_Records'] /
    missing_by_variant['Total_Records'] * 100
)

display(missing_by_variant)

,Total_Records,Missing_Records,Missing_Percentage
Variant,,,
Control,9886,101,1.021647
Treatment,9757,68,0.696936


In [35]:
# Compare app crashes for missing and non-missing podcast minutes

crash_missing_analysis = df1.assign(
    Podcast_Missing=df1['Podcast_Minutes_Listened'].isna()
).groupby('Podcast_Missing').agg(
    Total_Records=('App_Crashes', 'count'),
    Average_Crashes=('App_Crashes', 'mean'),
    Total_Crashes=('App_Crashes', 'sum')
)

display(crash_missing_analysis)

,Total_Records,Average_Crashes,Total_Crashes
Podcast_Missing,,,
False,19493,0.025086,489
True,170,0.035294,6


### 2.4 Investigate unmatched User ID

In [36]:
# Find activity records without a matching User_ID in the user map

unmatched_users = df1[
    ~df1['User_ID'].isin(df2['User_ID'])
]

print("Unmatched activity records:", len(unmatched_users))

print(
    "Unique unmatched User IDs:",
    unmatched_users['User_ID'].nunique()
)

display(unmatched_users.head(20))

Unmatched activity records: 20
Unique unmatched User IDs: 20


,Date,User_ID,Podcast_Minutes_Listened,Music_Minutes_Listened,Discovery_Feed_Clicks,App_Crashes,Missing_Reason
684,2025-03-03,U1000880,2.4,121.0,1,0,Not Missing
960,2025-03-08,U1000683,NaN,39.6,1,0,Originally Missing
1785,2025-03-16,U1000106,0.0,121.7,1,0,Not Missing
3445,2025-03-07,U1000951,32.0,154.1,2,0,Not Missing
3599,2025-03-14,U1000976,4.4,25.8,2,0,Not Missing
4822,2025-03-08,U1000631,13.3,0.0,0,0,Not Missing
5472,2025-03-11,U1000845,0.0,0.0,0,0,Not Missing
7649,2025-03-15,U1000364,0.0,89.2,3,0,Not Missing
8315,2025-03-08,U1000513,2.5,28.0,0,0,Not Missing
9213,2025-03-03,U1000195,27.2,199.2,4,0,Not Missing


### 2.5 Check Missing values of Acount Age column

In [37]:
# Identify users with missing account age
missing_age = df2[
    df2['Account_Age_Days'].isna()
]

print("Missing Account Age records:", len(missing_age))

# Check their experiment group
print("\nMissing Account Age by Variant:")
print(missing_age['Variant'].value_counts(dropna=False))

# Display the affected records
display(missing_age.head(30))

Missing Account Age records: 30

Missing Account Age by Variant:
Variant
Control      18
Treatment    12
Name: count, dtype: int64


,User_ID,Variant,Region,Premium_Status,Account_Age_Days
33,U595214,Control,Europe,Free,NaN
44,U493834,Treatment,Europe,Free,NaN
61,U104232,Control,LATAM,Free,NaN
79,U579154,Control,Europe,Premium,NaN
139,U487545,Treatment,APAC,Free,NaN
161,U351728,Control,North America,Premium,NaN
225,U258605,Control,APAC,Free,NaN
236,U493130,Control,Europe,Free,NaN
242,U181250,Control,Europe,Free,NaN
340,U613875,Control,North America,Premium,NaN


### 2.6 Standardize Region names

In [38]:
# Check region values before cleaning
print("Region values before cleaning:")
print(df2['Region'].value_counts(dropna=False))

Region values before cleaning:
Region
North America      613
Europe             579
APAC               342
LATAM              300
MEA                119
 north america      13
EUROPE              10
 europe              9
NORTH AMERICA        7
 latam               3
 mea                 3
 apac                2
Name: count, dtype: int64


In [39]:
# Remove extra spaces and convert region names to lowercase
df2['Region'] = (
    df2['Region']
    .astype('string')
    .str.strip()
    .str.lower()
)

# Check region values after cleaning
print("\nRegion values after cleaning:")
print(df2['Region'].value_counts(dropna=False))


Region values after cleaning:
Region
north america    633
europe           598
apac             344
latam            303
mea              122
Name: count, dtype: Int64


### 2.7 Validate data types and dates

In [41]:
print("Activity dataset data types:")
print(df1.dtypes)

print("User map data types:")
print(df2.dtypes)

Activity dataset data types:
Date                            str
User_ID                         str
Podcast_Minutes_Listened    float64
Music_Minutes_Listened      float64
Discovery_Feed_Clicks         int64
App_Crashes                   int64
Missing_Reason                  str
dtype: object
User map data types:
User_ID                 str
Variant                 str
Region               string
Premium_Status          str
Account_Age_Days    float64
dtype: object


In [42]:
# Convert Date column to datetime
df1['Date'] = pd.to_datetime(
    df1['Date'],
    errors='coerce'
)

# Verify conversion
print(df1['Date'].dtype)

datetime64[us]


In [43]:
print("Earliest date:", df1['Date'].min())
print("Latest date:", df1['Date'].max())

print("Unique dates:", df1['Date'].nunique())

print("Missing dates:", df1['Date'].isna().sum())

Earliest date: 2025-03-03 00:00:00
Latest date: 2025-03-16 00:00:00
Unique dates: 14
Missing dates: 0


## Cleaning Summary

In [44]:
# DATA CLEANING SUMMARY - ACTIVITY DATASET

activity_summary = pd.DataFrame({
    'Metric': [
        'Total Records',
        'Unique Users',
        'Duplicate Records',
        'Missing Podcast Minutes',
        'Missing Music Minutes',
        'Missing Dates',
        'Impossible Podcast Values',
        'Impossible Music Values',
        'Negative Podcast Values',
        'Negative Music Values'
    ],

    'Before Cleaning': [
        len(df1_original),
        df1_original['User_ID'].nunique(),
        df1_original.duplicated().sum(),
        df1_original['Podcast_Minutes_Listened'].isna().sum(),
        df1_original['Music_Minutes_Listened'].isna().sum(),
        df1_original['Date'].isna().sum(),
        (df1_original['Podcast_Minutes_Listened'] > 1440).sum(),
        (df1_original['Music_Minutes_Listened'] > 1440).sum(),
        (df1_original['Podcast_Minutes_Listened'] < 0).sum(),
        (df1_original['Music_Minutes_Listened'] < 0).sum()
    ],

    'After Cleaning': [
        len(df1),
        df1['User_ID'].nunique(),
        df1.duplicated().sum(),
        df1['Podcast_Minutes_Listened'].isna().sum(),
        df1['Music_Minutes_Listened'].isna().sum(),
        df1['Date'].isna().sum(),
        (df1['Podcast_Minutes_Listened'] > 1440).sum(),
        (df1['Music_Minutes_Listened'] > 1440).sum(),
        (df1['Podcast_Minutes_Listened'] < 0).sum(),
        (df1['Music_Minutes_Listened'] < 0).sum()
    ]
})

display(activity_summary)

,Metric,Before Cleaning,After Cleaning
0,Total Records,19741,19663
1,Unique Users,2020,2020
2,Duplicate Records,78,0
3,Missing Podcast Minutes,158,170
4,Missing Music Minutes,0,0
5,Missing Dates,0,0
6,Impossible Podcast Values,13,0
7,Impossible Music Values,0,0
8,Negative Podcast Values,0,0
9,Negative Music Values,0,0


In [45]:
# DATA CLEANING SUMMARY - USER DATASET

user_summary = pd.DataFrame({
    'Metric': [
        'Total Users',
        'Unique User IDs',
        'Duplicate Records',
        'Missing Account Age',
        'Missing Variant',
        'Missing Region',
        'Missing Premium Status'
    ],

    'Before Cleaning': [
        len(df2_original),
        df2_original['User_ID'].nunique(),
        df2_original.duplicated().sum(),
        df2_original['Account_Age_Days'].isna().sum(),
        df2_original['Variant'].isna().sum(),
        df2_original['Region'].isna().sum(),
        df2_original['Premium_Status'].isna().sum()
    ],

    'After Cleaning': [
        len(df2),
        df2['User_ID'].nunique(),
        df2.duplicated().sum(),
        df2['Account_Age_Days'].isna().sum(),
        df2['Variant'].isna().sum(),
        df2['Region'].isna().sum(),
        df2['Premium_Status'].isna().sum()
    ]
})

display(user_summary)

,Metric,Before Cleaning,After Cleaning
0,Total Users,2000,2000
1,Unique User IDs,2000,2000
2,Duplicate Records,0,0
3,Missing Account Age,30,30
4,Missing Variant,0,0
5,Missing Region,0,0
6,Missing Premium Status,0,0


## Exporting cleaning summaries to csv

In [47]:
activity_summary.to_csv(
   r'C:\Spotify_A-B_Test_project\01_Data\02_processed_data\activity_summary.csv',
    index=False
)

user_summary.to_csv(
   r'C:\Spotify_A-B_Test_project\01_Data\02_processed_data\user_summary.csv',
    index=False
)

print("Cleaning summaries exported successfully!")

Cleaning summaries exported successfully!


## Creating cleaning log

In [48]:
import os
import pandas as pd

# Create processed folder if it doesn't exist
os.makedirs(r"C:\Spotify_A-B_Test_project\01_Data\02_processed_data", exist_ok=True)

# Create cleaning log
cleaning_log = pd.DataFrame([
    {
        "Dataset": "Daily User Activity",
        "Issue": "Exact duplicate records",
        "Records_Affected": 78,
        "Action_Taken": "Removed exact duplicate rows",
        "Reason": "Prevent duplicate activity from being counted twice"
    },
    {
        "Dataset": "Daily User Activity",
        "Issue": "Impossible podcast listening values",
        "Records_Affected": 12,
        "Action_Taken": "Replaced values above 1440 with NaN",
        "Reason": "Daily listening cannot exceed 1440 minutes"
    },
    {
        "Dataset": "Daily User Activity",
        "Issue": "Originally missing podcast minutes",
        "Records_Affected": 158,
        "Action_Taken": "Retained as NaN",
        "Reason": "Avoid assuming missing listening data means zero"
    },
    {
        "Dataset": "Daily User Activity",
        "Issue": "Unmatched User IDs",
        "Records_Affected": 20,
        "Action_Taken": "Flagged for investigation; retained in dataset",
        "Reason": "Experiment assignment could not be verified"
    },
    {
        "Dataset": "User Experiment Map",
        "Issue": "Missing Account Age",
        "Records_Affected": 30,
        "Action_Taken": "Retained as NaN",
        "Reason": "Avoid inventing user account ages"
    },
    {
        "Dataset": "User Experiment Map",
        "Issue": "Inconsistent Region formatting",
        "Records_Affected": 2000,
        "Action_Taken": "Converted region names to lowercase and removed spaces",
        "Reason": "Ensure consistent region categories"
    },
    {
        "Dataset": "Daily User Activity",
        "Issue": "Date data type",
        "Records_Affected": "Validated",
        "Action_Taken": "Converted Date to datetime",
        "Reason": "Support date-based analysis"
    }
])

display(cleaning_log)

,Dataset,Issue,Records_Affected,Action_Taken,Reason
0,Daily User Activity,Exact duplicate records,78,Removed exact duplicate rows,Prevent duplicate activity from being counted ...
1,Daily User Activity,Impossible podcast listening values,12,Replaced values above 1440 with NaN,Daily listening cannot exceed 1440 minutes
2,Daily User Activity,Originally missing podcast minutes,158,Retained as NaN,Avoid assuming missing listening data means zero
3,Daily User Activity,Unmatched User IDs,20,Flagged for investigation; retained in dataset,Experiment assignment could not be verified
4,User Experiment Map,Missing Account Age,30,Retained as NaN,Avoid inventing user account ages
5,User Experiment Map,Inconsistent Region formatting,2000,Converted region names to lowercase and remove...,Ensure consistent region categories
6,Daily User Activity,Date data type,Validated,Converted Date to datetime,Support date-based analysis


In [49]:
cleaning_log.to_csv(
    r"C:\Spotify_A-B_Test_project\01_Data\02_processed_data\cleaning_log.csv",
    index=False
)

print("Cleaning log saved successfully!")

Cleaning log saved successfully!


## Exporting Cleaned Dataset

In [50]:
# Export cleaned activity dataset
df1.to_csv(
    r"C:\Spotify_A-B_Test_project\01_Data\02_processed_data\cleaned_activity.csv",
    index=False
)

# Export cleaned user experiment map
df2.to_csv(
    r"C:\Spotify_A-B_Test_project\01_Data\02_processed_data\cleaned_users.csv",
    index=False
)

print("Cleaned datasets exported successfully!")

Cleaned datasets exported successfully!


In [51]:
print("Files in processed folder:")

for file in os.listdir("C:\\Spotify_A-B_Test_project\\01_Data\\02_processed_data"):
    print(file)

Files in processed folder:
activity_summary.csv
cleaned_activity.csv
cleaned_users.csv
cleaning_log.csv
user_summary.csv


### Data Cleaning Conclusion

The data cleaning process was completed to improve the consistency and reliability of the datasets before conducting the A/B test analysis.

The following actions were performed:

- Removed 78 duplicate activity records.
- Identified 12 impossible podcast listening values (greater than 1,440 minutes per day) and converted them to missing values.
- Retained missing podcast listening values rather than replacing them with zero or an average.
- Identified 20 activity records with User IDs that did not match the experiment user map. These records were retained but flagged for investigation.
- Retained 30 missing values in Account_Age_Days.
- Standardized the Region column and converted the Date column to datetime format.
- Exported the cleaned datasets and cleaning log for further analysis.

**Important considerations:** Missing podcast listening values and unmatched User IDs require further investigation. These issues must be considered when calculating the primary outcome and interpreting the experiment results.

Overall, the datasets have undergone initial cleaning and validation and are ready for the next stage of data quality assessment and preparation for A/B test analysis.